In [ ]:
import torch
assert torch.cuda.is_available(), 'NO GPU - aborting before slow CPU load'
print('GPU OK:', torch.cuda.get_device_name(0))
import subprocess
subprocess.run(['nvidia-smi'], check=False)
import sys
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','transformers==5.19.0'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','sentence-transformers==6.1.0'], check=False)
import json, os, re, time, glob, collections
import numpy as np
from sentence_transformers import SentenceTransformer
print('imports OK', flush=True)

In [ ]:
# ---- discover inputs (dataset mount) ----
import glob
D = glob.glob('/kaggle/input/**/curated_actions_train.jsonl', recursive=True)
print('found train:', D, flush=True)
TRAIN = D[0]
TEST = glob.glob('/kaggle/input/**/curated_actions_test.jsonl', recursive=True)[0]
ACTIONS = ['click','swipe','open_app','wait','input_text','navigate_back','long_press']
PREFIX = 'task: classification | query: '
print('TRAIN', TRAIN, '| TEST', TEST, flush=True)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
class Model(nn.Module):
    def __init__(self, dim, K):
        super().__init__()
        self.action = nn.Linear(dim, K)
        self.calib = nn.Sequential(nn.Linear(2,16), nn.ReLU(), nn.Linear(16,1))
    def forward(self, S):
        logits = self.action(S)
        probs = torch.softmax(logits, 1)
        t2 = torch.topk(probs,2,1).values
        margin = (t2[:,0]-t2[:,1]).unsqueeze(1)
        ci = torch.cat([probs.max(1,keepdim=True).values, margin], 1)
        p_success = torch.sigmoid(self.calib(ci)).squeeze(1)
        return logits, p_success

def risk_gate(conf, correct, taus):
    rows=[]
    for tau in taus:
        act = conf >= tau
        n_act = int(act.sum())
        cov = n_act/len(conf)
        risk = float((~correct[act]).mean()) if n_act>0 else 0.0
        rows.append(dict(tau=round(float(tau),3), coverage=round(cov,4), n_act=n_act, risk=round(risk,4)))
    return rows

SAVE='/kaggle/working'

In [ ]:
import torch, numpy as np, time
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('google/embeddinggemma-2', model_kwargs={'torch_dtype': torch.bfloat16}, config_kwargs={'audio_config': None})
m.eval().to('cuda')
print('model load ok', flush=True)

def encode(rows):
    txt = [PREFIX + (r['goal'] or '') + ' [step: ' + (r['step_instruction'] or '') + ']' for r in rows]
    return np.asarray(m.encode(txt, convert_to_numpy=True, normalize_embeddings=True), dtype=np.float32)

train = [json.loads(l) for l in open(TRAIN)]
test  = [json.loads(l) for l in open(TEST)]
print('encoding text...', flush=True)
S_tr = encode(train); S_te = encode(test)
y_tr = np.array([ACTIONS.index(r['action']) if r['action'] in ACTIONS else ACTIONS.index('click') for r in train], np.int64)
y_te = np.array([ACTIONS.index(r['action']) if r['action'] in ACTIONS else ACTIONS.index('click') for r in test], np.int64)
np.save(SAVE+'/S_tr.npy', S_tr); np.save(SAVE+'/S_te.npy', S_te)
np.save(SAVE+'/y_tr.npy', y_tr); np.save(SAVE+'/y_te.npy', y_te)
print('emb', S_tr.shape, S_te.shape, flush=True)

In [ ]:
import torch, numpy as np, time
DEV='cuda'; K=len(ACTIONS)
S_tr=np.load(SAVE+'/S_tr.npy'); S_te=np.load(SAVE+'/S_te.npy')
y_tr=np.load(SAVE+'/y_tr.npy'); y_te=np.load(SAVE+'/y_te.npy')
results=[]
for objective in ('proper','ce'):
    torch.manual_seed(0); np.random.seed(0)
    model=Model(S_tr.shape[1],K).to(DEV)
    opt=torch.optim.Adam(model.parameters(), lr=2e-3)
    ce=nn.CrossEntropyLoss()
    St=torch.from_numpy(S_tr).to(DEV); yt=torch.from_numpy(y_tr).to(torch.long).to(DEV)
    Se=torch.from_numpy(S_te).to(DEV); ye=torch.from_numpy(y_te).to(torch.long).to(DEV)
    t0=time.time()
    for ep in range(60):
        model.train()
        logits,ps=model(St)
        correct=(logits.argmax(1)==yt).float()
        loss=ce(logits,yt)
        if objective in ('proper','joint'):
            loss=loss+0.5*torch.mean((ps-correct)**2)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        logits,ps=model(Se)
        pred=logits.argmax(1)
        acc=(pred==ye).float().mean().item()
        correct=(pred==ye).cpu().numpy().astype(bool)
    conf = ps.cpu().numpy() if objective=='proper' else torch.softmax(logits,1).max(1).values.cpu().numpy()
    print(f'=== {objective}: base_acc={acc:.4f} conf_range=[{conf.min():.3f},{conf.max():.3f}] mean={conf.mean():.3f}', flush=True)
    taus=np.round(np.arange(0.40,1.0001,0.05),3)
    rows=risk_gate(conf, correct, taus)
    for r in rows:
        r['objective']=objective; r['base_acc']=round(acc,4)
        results.append(r)
    for r in rows:
        print(f"  {objective} tau={r['tau']:.2f} cov={r['coverage']:.3f} n_act={r['n_act']:5d} risk={r['risk']:.3f}", flush=True)
    print(flush=True)
print('=== SUMMARY ===')
import pandas as pd
pd.DataFrame(results).to_csv(SAVE+'/g4_riskgate.csv', index=False)
print(pd.DataFrame(results).to_string(index=False))
print('DONE', flush=True)